# Phase 3.10b — Full BBA Training with GraphTransformerScoreModel

Clones `grapentt/riemannian-scoremd` from GitHub, **verifies precomputed data
integrity against a bit-exact sha256 manifest**, trains the graph transformer
on all 62k BBA frames, and **saves every artefact (loss history, checkpoints,
env metadata, eval JSON) under a per-run timestamped folder on Drive**.

**Before running**: select an A100 GPU runtime — `Runtime → Change runtime type → A100`.
(Metadata defaults to `gpuType=A100`, but Colab may downgrade silently if A100 is unavailable.)

**Expected**: ~11 min training on A100 at ~0.22 ms/step. Loss drops from ~22 → <5
by epoch 3000. Success gate: held-out cos_sim > **0.95**.

**Artefacts produced** (under `DRIVE_OUT_DIR/run_<UTC_TS>/`):
- `env_info.json` (jax/flax/jaxlib versions, git SHA, GPU)
- `config.json` (every hyperparam)
- `loss_history.json` (epoch, smoothed loss)
- `310_bba_gt_results.npz` (bundled arrays for offline plotting)
- `310_loss_curve.png`
- `eval.json` (cos_sim overall + per-t-bin + PASS/FAIL on Phase 3.10b gate)
- `checkpoints/ckpt_epoch_NNNNNN.npz` (every `LOG_EVERY` epochs)
- `checkpoints/ckpt_final.npz`

## 0. GPU check + install deps

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
gpu_info = result.stdout.strip()
print("GPU  :", gpu_info or "NOT FOUND — change runtime type!")

# Phase 3.10b expects an A100 (recommended). Colab may silently downgrade if A100
# capacity is unavailable. Warn if we did not get one — does not block, since T4
# can still run the job (slower, larger OOM risk at full batch).
if gpu_info and "A100" not in gpu_info.upper():
    print(f"\n⚠️  WARNING: Not running on A100. Got: {gpu_info}")
    print("   Phase 3.10b was sized for A100. T4 (16 GB) may OOM at batch=64.")
    print("   Consider Runtime → Change runtime type → A100, or reduce BATCH_SIZE.")

import jax
print("\nJAX  :", jax.__version__)
print("Devs :", jax.devices())

In [ ]:
# Pin JAX/jaxlib/Flax to match local .venv (jax 0.10.0, jaxlib 0.10.0, flax 0.10.4)
# so the Colab runtime matches the version where the precomputed targets were generated
# and where parity tests pass. Without this pin, Colab's default JAX (often ahead of
# what the codebase supports) breaks ShapeManifold + GraphTransformerScoreModel imports.
#
# We install into /content/pyenv (a venv-like overlay) and prepend it to sys.path so it
# wins over Colab's system dist-packages WITHOUT a kernel restart.
import subprocess, sys, importlib

TARGET = "/content/pyenv"

# Detect GPU vs CPU — install matching jaxlib build.
import os
HAS_GPU = "GPU" in subprocess.run(
    ["bash", "-c", "command -v nvidia-smi && nvidia-smi -L || true"],
    capture_output=True, text=True,
).stdout.upper()

jax_spec = (
    'jax[cuda12]==0.10.0' if HAS_GPU else 'jax==0.10.0'
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--target", TARGET,
     "--upgrade",
     jax_spec,
     "flax==0.10.4",
     "einops",
     "orbax-checkpoint>=0.11.20",
     # diffrax pinned at the version compatible with jax 0.10 — leave unpinned for now,
     # add a pin here if a future Colab pull breaks it.
     ],
    check=True,
)

# Prepend so these packages shadow the system ones for the rest of the run.
if TARGET not in sys.path:
    sys.path.insert(0, TARGET)

# Evict any already-imported JAX/Flax modules so the prepended ones win.
for mod_name in list(sys.modules):
    if (
        mod_name == "jax" or mod_name.startswith("jax.")
        or mod_name == "jaxlib" or mod_name.startswith("jaxlib.")
        or mod_name == "flax" or mod_name.startswith("flax.")
        or mod_name == "orbax" or mod_name.startswith("orbax.")
    ):
        del sys.modules[mod_name]

# Re-import and verify
import jax, flax
jax_path = jax.__file__
flax_path = flax.__file__
print(f"JAX   : {jax.__version__}     ({jax_path})")
print(f"Flax  : {flax.__version__}    ({flax_path})")
assert jax.__version__ == "0.10.0", f"JAX version mismatch: {jax.__version__} (expected 0.10.0)"
assert flax.__version__.startswith("0.10."), f"Flax version mismatch: {flax.__version__} (expected 0.10.x)"
print("✅ Versions match local .venv — safe to proceed.")

## 1. Clone repo from GitHub

In [16]:
import os, subprocess

REPO_DIR = "/content/riemannian-scoremd"
GITHUB_URL = "https://github.com/grapentt/riemannian-scoremd.git"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes.")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {GITHUB_URL} ...")
    subprocess.run(["git", "clone", GITHUB_URL, REPO_DIR], check=True)

# Verify key source files are present
for rel in [
    "src/manifold/pointcloud_jax.py",
    "src/diffusion/manifold_sde.py",
    "src/models/graph_transformer_jax.py",
    "src/training/train_manifold.py",
]:
    path = os.path.join(REPO_DIR, rel)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {status}  {rel}")

# Add src/ to Python path
SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("\nRepo ready.")

Repo already cloned — pulling latest changes.
  OK  src/manifold/pointcloud_jax.py
  OK  src/diffusion/manifold_sde.py
  OK  src/models/graph_transformer_jax.py
  OK  src/training/train_manifold.py

Repo ready.


## 2. Mount Drive and configure paths

**Edit `DRIVE_DATA_DIR` and `DRIVE_OUT_DIR`** to match where you stored the
precomputed BBA data on your Drive (`noised_r00.npz` … `noised_r09.npz`).
Results (checkpoints, loss curve, eval metrics) will be written to `DRIVE_OUT_DIR`.

In [17]:
from google.colab import drive
drive.mount("/content/drive")

# ── EDIT THESE ─────────────────────────────────────────────────────────────
# Folder on Drive that contains noised_r00.npz … noised_r09.npz
DRIVE_DATA_DIR = "/content/drive/MyDrive/thesis-data/precomputed/bba"

# Where to write checkpoints and results on Drive
DRIVE_OUT_DIR  = "/content/drive/MyDrive/thesis-data/runs/bba_gt_phase310"
# ───────────────────────────────────────────────────────────────────────────

# Local scratch dirs (fast SSD on Colab VM)
LOCAL_DATA_DIR = "/content/data/precomputed/bba"
LOCAL_CKPT_DIR = "/content/checkpoints/bba_gt_phase310"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR,  exist_ok=True)

# Verify precomputed files exist on Drive
import glob
drive_files = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, "noised_r*.npz")))
print(f"Found {len(drive_files)} precomputed files on Drive:")
for f in drive_files:
    print(f"  {os.path.basename(f)}")
assert len(drive_files) > 0, f"No noised_r*.npz files found in {DRIVE_DATA_DIR}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 10 precomputed files on Drive:
  noised_r00.npz
  noised_r01.npz
  noised_r02.npz
  noised_r03.npz
  noised_r04.npz
  noised_r05.npz
  noised_r06.npz
  noised_r07.npz
  noised_r08.npz
  noised_r09.npz


## 2a. Verify Drive data integrity (sha256)

The 10 `noised_r*.npz` files on Drive must be bit-for-bit identical to the local
`data/precomputed/bba/` copies that were generated by the post-transpose-fix
pipeline (see `HISTORY.md` 2026-05-28 entry). This cell hashes each file on
Drive and asserts against a manifest baked from the local repository on
2026-06-29. **Hard-fails** if any hash diverges — that means the Drive copy
predates the transpose fix and would silently train on geometrically wrong
targets (the exact failure mode of Phase 3.6, see `EXPERIMENTS_LOG.md` E001).

In [ ]:
# Bit-exact hash manifest captured locally on 2026-06-29.
# Generated with: shasum -a 256 data/precomputed/bba/noised_r*.npz
# If you re-precompute the data on Drive, REPLACE this dict — old hashes
# will FAIL even if the new data is correct.
EXPECTED_SHA256 = {
    "noised_r00.npz": "a80dc340f349a6ea3c6e8ba083d62499c92315550e0e6d9c5f672e6ec7f9609e",
    "noised_r01.npz": "36b9236f248a9378a765c554421a1b1fea1905bece79733ada95dedf60fa47fe",
    "noised_r02.npz": "4c548ac852a94c15d66d1fba9cbe94dbcc417e706660f9f0757b141ca49b63dd",
    "noised_r03.npz": "0fcfb3e5d50647f0a0ea44d4caa1d9dd49c1a352486f5c26180c9a4ed10dcdfa",
    "noised_r04.npz": "457f36c6a998a3016a5ad356efe315106deaae35e6d3707490161a786243cdb9",
    "noised_r05.npz": "6ff2d466a1a5dd570d953a34ba7d193ab36d0a5a1d9f70f21c339ca4222e6c1b",
    "noised_r06.npz": "ec903e1462adcae7b303bc9bfd21dabf077ea735f60025e111a84b433b4545dd",
    "noised_r07.npz": "7d6bdd57e18fdcd419c0f448745306855412d2418752f8fc89be58aea37a351b",
    "noised_r08.npz": "f1e323c690652aec6048c219852299780f9836508269ad2fddac0e9e2bd218dd",
    "noised_r09.npz": "080e6f3fc836e610f26f21bcfcd535f380fdb5be1719b42195e751628692c60e",
}

import hashlib

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            blk = fh.read(chunk)
            if not blk:
                break
            h.update(blk)
    return h.hexdigest()

mismatches = []
print(f"Verifying {len(drive_files)} files on Drive against local manifest…")
for f in drive_files:
    name = os.path.basename(f)
    expected = EXPECTED_SHA256.get(name)
    if expected is None:
        print(f"  ⚠  {name}: no expected hash (extra file?) — skipping")
        continue
    got = _sha256(f)
    if got == expected:
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name}\n      expected {expected}\n      got      {got}")
        mismatches.append(name)

# Also ensure no expected file is missing on Drive.
missing = [name for name in EXPECTED_SHA256 if not any(name == os.path.basename(f) for f in drive_files)]
for name in missing:
    print(f"  ✗ {name}: MISSING from Drive")

if mismatches or missing:
    raise RuntimeError(
        f"SHA256 verification FAILED: {len(mismatches)} mismatched, {len(missing)} missing.\n"
        "Drive precomputed data does NOT match the local post-transpose-fix copy.\n"
        "Refusing to train — see HISTORY.md 2026-05-28 entry for the failure mode."
    )
print(f"\n✅ All {len(EXPECTED_SHA256)} files match the local manifest — safe to train.")

## 2b. (Optional) Regenerate precomputed data — SKIP if data is on Drive

**✅ Data already on Drive (`thesis-data/precomputed/bba/noised_r00…r09.npz`)
— skip both cells in this section and continue at step 3.**

Only run this if Drive data is missing or stale (e.g. after changing `alpha` or
the `s_log` implementation). Each repeat takes ~8 min on Colab CPU; 10 repeats
≈ 80 min. The cell is a no-op if all 10 files already exist on Drive.
Results are pushed back to `DRIVE_DATA_DIR` automatically.

In [ ]:
# ── (Optional) Regenerate precomputed BBA data ──────────────────────────────
# NO-OP if all 10 files already exist on Drive.
# Only regenerate if Drive data is missing or the s_exp/s_log implementation changed.
#
# alpha=1.0 is correct for BBA and AK. s_log targets (Riemannian gradient).
# Expected time if regenerating: ~8 min/repeat × 10 repeats ≈ 80 min (CPU).

import subprocess, os, glob, shutil

N_REPEATS = 10

# Check how many files are already on Drive
existing = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, "noised_r*.npz")))
if len(existing) == N_REPEATS:
    print(f"✅ All {N_REPEATS} precomputed files already on Drive — nothing to do.")
    for f in existing:
        print(f"  {os.path.basename(f)}")
else:
    missing = N_REPEATS - len(existing)
    print(f"⚠️  {missing} files missing on Drive — regenerating all {N_REPEATS} repeats.")

    cmd = [
        "python", os.path.join(REPO_DIR, "scripts", "precompute_noised_data.py"),
        "--protein", "bba",
        "--n-repeats", str(N_REPEATS),
        "--output", "/content/data/precomputed",  # script appends /bba
        "--seed", "0",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    gen_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "noised_r*.npz")))
    assert len(gen_files) == N_REPEATS, f"Expected {N_REPEATS}, got {len(gen_files)}"

    # Push to Drive
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for f in gen_files:
        dst = os.path.join(DRIVE_DATA_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f"  → Drive: {os.path.basename(f)}")
    print(f"\n✅ All {N_REPEATS} files saved to {DRIVE_DATA_DIR}")
    print("Download locally via Drive desktop sync or rclone.")


In [18]:
# Copy precomputed files to local SSD for fast access during training.
# Drive I/O is slow (~50 MB/s); this copy takes ~30s and makes training 3-5× faster.
import shutil, time

t0 = time.time()
for src_f in drive_files:
    dst_f = os.path.join(LOCAL_DATA_DIR, os.path.basename(src_f))
    if not os.path.exists(dst_f):
        shutil.copy2(src_f, dst_f)
        print(f"  Copied {os.path.basename(src_f)}")
    else:
        print(f"  Already present: {os.path.basename(src_f)}")

local_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "noised_r*.npz")))
print(f"\nLocal SSD: {len(local_files)} files  ({time.time()-t0:.0f}s)")

  Already present: noised_r00.npz
  Already present: noised_r01.npz
  Already present: noised_r02.npz
  Already present: noised_r03.npz
  Already present: noised_r04.npz
  Already present: noised_r05.npz
  Already present: noised_r06.npz
  Already present: noised_r07.npz
  Already present: noised_r08.npz
  Already present: noised_r09.npz

Local SSD: 10 files  (0s)


## 3. Imports and sanity checks

In [ ]:
import os, glob, sys

# Evict any flax modules already cached from before the pip upgrade
# (needed when running cells without a kernel restart)
to_evict = [k for k in sys.modules if k == 'flax' or k.startswith('flax.')]
for k in to_evict:
    del sys.modules[k]

import jax
import jax.numpy as jnp
import numpy as np

from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP
from models.graph_transformer_jax import GraphTransformerScoreModel
from training.train_manifold import train_from_precomputed

print("All imports OK")

# local_files: all precomputed repeats on the local SSD
# (populated by the copy-data cell; redefined here in case cells ran out of order)
local_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "noised_r*.npz")))
assert len(local_files) > 0, (
    f"No noised_r*.npz found in {LOCAL_DATA_DIR}.\n"
    "Run the 'copy-data' cell (step 2) first to copy files from Drive to local SSD."
)

# Quick data sanity check
sample = np.load(local_files[0])
print(f"\nData shape (r00):")
print(f"  x_t    : {sample['x_t'].shape}  {sample['x_t'].dtype}")
print(f"  s_true : {sample['s_true'].shape}")
print(f"  t      : {sample['t'].shape}  range [{sample['t'].min():.2f}, {sample['t'].max():.2f}]")
nan_xt = np.isnan(sample['x_t']).sum()
nan_st = np.isnan(sample['s_true']).sum()
print(f"  NaN x_t: {nan_xt}   NaN s_true: {nan_st}")

## 4. Configuration

In [ ]:
# ── Manifold ─────────────────────────────────────────────────────────────────
# BBA: 28 Cα atoms, d=3, manifold dim = 3*28 - 6 = 78
N_ATOMS = 28
D       = 3
ALPHA   = 1.0     # w^delta gyration weight (both BBA and AK use 1.0; ShapeManifold default is 0.1 but never used for proteins)

# ── Training ─────────────────────────────────────────────────────────────────
N_EPOCHS   = 3000
BATCH_SIZE = 64
LR         = 1e-3
GRAD_CLIP  = 1.0
EMA_DECAY  = 0.995
LOG_EVERY  = 50
SEED       = 42

# eps_parameterization: predict v_h_unit (unit-g-norm noise direction)
# instead of the raw score. ||v_h_unit||_E ≈ 2.81 constant across t,
# eliminating the 200× amplitude variation that caused the zero-output basin.
# See TRAINING_COLLAPSE_ANALYSIS.md for the full root-cause analysis.
EPS_PARAM = True

# conservative=True: score=-grad_x E_theta (energy model, nested autodiff, ~350ms/step CPU)
# conservative=False: direct score output from node embeddings (~2x faster; same cos_sim)
# Run 3.10a ablation first to confirm; default to False for full training.
CONSERVATIVE = False

# ── Model (ScoreMD BBA 'large_potential' config) ──────────────────────────────
HIDDEN_DIM  = 128
NUM_LAYERS  = 3
NUM_HEADS   = 8
DIM_HEAD    = 64
INPUT_SCALE = 6.26   # BBA mean gyration radius ≈ 6.26 Å

print("Config:")
print(f"  protein      : BBA  ({N_ATOMS} Cα, manifold dim={N_ATOMS*D - D*(D+1)//2})")
print(f"  model        : GraphTransformerScoreModel  hidden={HIDDEN_DIM} layers={NUM_LAYERS} heads={NUM_HEADS} dim_head={DIM_HEAD}")
print(f"  epochs       : {N_EPOCHS}")
print(f"  batch        : {BATCH_SIZE}")
print(f"  lr           : {LR}  (cosine → 1e-5)")
print(f"  eps_param    : {EPS_PARAM}")
print(f"  data         : {LOCAL_DATA_DIR}")
print(f"  checkpoints  : {LOCAL_CKPT_DIR}")

## 5. Initialise manifold, SDE, and model

In [ ]:
# ShapeManifold constructor: dim=d, numpoints=n, alpha=delta
# (no base_point needed for training — only needed for alignment in s_geodesic/s_exp)
manifold = ShapeManifold(dim=D, numpoints=N_ATOMS, alpha=ALPHA)
sde      = ManifoldVP(manifold)

model = GraphTransformerScoreModel(
    n=N_ATOMS,
    d=D,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dim_head=DIM_HEAD,
    input_scale=INPUT_SCALE,
    conservative=CONSERVATIVE,
)

# Count parameters
rng     = jax.random.PRNGKey(SEED)
dummy_x = jnp.zeros((1, N_ATOMS * D))
dummy_t = jnp.zeros((1, 1))
params_tmp = model.init(rng, dummy_x, dummy_t)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params_tmp))
print(f"Model parameters: {n_params:,}")
# Expected: ~1,393,409

## 6. Step-time benchmark (estimates total training time)

In [ ]:
import time, optax
from training.train_manifold import make_train_step

optimizer_bm = optax.chain(
    optax.clip_by_global_norm(GRAD_CLIP),
    optax.adam(LR),
)
step_fn = make_train_step(
    model, manifold, sde, optimizer_bm,
    likelihood_weighting=False,
    eps_parameterization=EPS_PARAM,
)

rng_bm = jax.random.PRNGKey(99)
p_bm   = model.init(rng_bm, dummy_x, dummy_t)
ep_bm  = p_bm
os_bm  = optimizer_bm.init(p_bm)

B = BATCH_SIZE
xb = jax.random.normal(rng_bm, (B, N_ATOMS, D))
sb = jax.random.normal(rng_bm, (B, N_ATOMS, D))
tb = jax.random.uniform(rng_bm, (B,), minval=0.1, maxval=0.9)

# Warmup JIT
p_bm, ep_bm, os_bm, loss_bm = step_fn(p_bm, ep_bm, os_bm, xb, sb, tb)
loss_bm.block_until_ready()
print(f"JIT warmup OK  (loss={float(loss_bm):.2f})")

N_BM = 40
t0 = time.perf_counter()
acc = jnp.zeros(())
for _ in range(N_BM):
    p_bm, ep_bm, os_bm, loss_bm = step_fn(p_bm, ep_bm, os_bm, xb, sb, tb)
    acc = acc + loss_bm
acc.block_until_ready()
ms_per_step = (time.perf_counter() - t0) / N_BM * 1000

N_frames = sample['x_t'].shape[0]
steps_per_epoch = max(1, N_frames // BATCH_SIZE)
est_min = N_EPOCHS * steps_per_epoch * ms_per_step / 1000 / 60

print(f"\nStep time       : {ms_per_step:.2f} ms/step")
print(f"Steps/epoch     : {steps_per_epoch}  ({N_frames} frames / {BATCH_SIZE} batch)")
print(f"Est. total time : {est_min:.1f} min  ({N_EPOCHS} epochs)")

## 7. Training

`train_from_precomputed`:
- Preloads all repeat files into device RAM (one copy of ~600 MB total)
- Cycles repeats across epochs for stochasticity
- Accumulates loss on device, syncs only at `log_every` — no GPU→CPU stall per step
- Saves `.npz` checkpoints every `log_every` epochs to `LOCAL_CKPT_DIR`

In [ ]:
# Training run — with incremental Drive sync after every log_every epochs.
#
# Drive sync strategy:
#   * train_from_precomputed already writes ckpt_<epoch>.npz to LOCAL_CKPT_DIR
#     every LOG_EVERY epochs.
#   * We background a sync thread that copies new checkpoints + appends the
#     latest loss history to Drive incrementally, so if Colab kills the runtime
#     mid-training we recover from the last checkpoint AND have the loss curve
#     up to that point.
#   * Final save (cell 10) is the canonical persisted run; this is a safety net.

import os, json, time, threading, shutil, glob, subprocess, sys
from datetime import datetime, timezone

# Run identity — every training run gets a unique, timestamped subdir on Drive.
RUN_ID    = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DRIVE_RUN_DIR = os.path.join(DRIVE_OUT_DIR, f"run_{RUN_ID}")
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_RUN_DIR, "checkpoints"), exist_ok=True)
print(f"Run ID         : {RUN_ID}")
print(f"Drive run dir  : {DRIVE_RUN_DIR}")

# Capture environment for reproducibility — written before training starts so we
# have it even if the run crashes.
try:
    git_sha = subprocess.run(
        ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except Exception as e:
    git_sha = f"unknown ({e})"

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip() or "CPU"
except Exception:
    gpu_name = "unknown"

import jax, flax
env_info = {
    "run_id"       : RUN_ID,
    "git_sha"      : git_sha,
    "git_remote"   : GITHUB_URL,
    "jax_version"  : jax.__version__,
    "jaxlib_version": getattr(__import__("jaxlib"), "__version__", "unknown"),
    "flax_version" : flax.__version__,
    "gpu"          : gpu_name,
    "python"       : sys.version.split()[0],
    "started_utc"  : datetime.now(timezone.utc).isoformat(),
}
config_info = dict(
    n_atoms=N_ATOMS, d=D, alpha=ALPHA,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    grad_clip=GRAD_CLIP, ema_decay=EMA_DECAY, log_every=LOG_EVERY, seed=SEED,
    eps_param=EPS_PARAM, conservative=CONSERVATIVE,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
    dim_head=DIM_HEAD, input_scale=INPUT_SCALE,
)
with open(os.path.join(DRIVE_RUN_DIR, "env_info.json"), "w") as f:
    json.dump(env_info, f, indent=2)
with open(os.path.join(DRIVE_RUN_DIR, "config.json"), "w") as f:
    json.dump(config_info, f, indent=2)
print(f"  env_info + config written to Drive")
print(f"  git SHA: {git_sha[:12]}")

# ── Background incremental Drive sync ──────────────────────────────────────────
_stop_sync = threading.Event()
def _drive_sync_loop():
    """Every 30s: rsync local checkpoints + latest loss_history to Drive."""
    seen = set()
    while not _stop_sync.wait(30.0):
        try:
            # Sync any new checkpoints
            for src in sorted(glob.glob(os.path.join(LOCAL_CKPT_DIR, "ckpt_*.npz"))):
                if src in seen:
                    continue
                dst = os.path.join(DRIVE_RUN_DIR, "checkpoints", os.path.basename(src))
                shutil.copy2(src, dst)
                seen.add(src)
        except Exception:
            pass  # don't crash the sync thread; final save will catch up
_sync_thread = threading.Thread(target=_drive_sync_loop, daemon=True)
_sync_thread.start()
print("Background Drive sync thread started (30s interval).\n")

# ── Run training (this is the long-running call) ──────────────────────────────
t_start = time.time()
try:
    state, loss_history = train_from_precomputed(
        model=model,
        manifold=manifold,
        sde=sde,
        precomputed_dir=LOCAL_DATA_DIR,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        grad_clip=GRAD_CLIP,
        ema_decay=EMA_DECAY,
        eps_parameterization=EPS_PARAM,
        seed=SEED,
        log_every=LOG_EVERY,
        ckpt_dir=LOCAL_CKPT_DIR,
    )
finally:
    _stop_sync.set()
    _sync_thread.join(timeout=5.0)
t_wall = time.time() - t_start

print(f"\nTraining complete in {t_wall/60:.2f} min.")
epochs_v, losses_v = zip(*loss_history)
print(f"  First loss : {losses_v[0]:.4f}  (epoch {epochs_v[0]})")
print(f"  Final loss : {losses_v[-1]:.4f}  (epoch {epochs_v[-1]})")

# Persist loss curve immediately to Drive (before eval / plot cells) so even a
# kernel disconnect now doesn't lose the loss history.
loss_history_json = [{"epoch": int(e), "loss": float(l)} for e, l in loss_history]
with open(os.path.join(DRIVE_RUN_DIR, "loss_history.json"), "w") as f:
    json.dump(loss_history_json, f, indent=2)
print(f"  loss_history.json → Drive ({len(loss_history)} entries)")

## 8. Loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs_arr = np.array([e for e, _ in loss_history])
losses_arr = np.array([l for _, l in loss_history])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_arr, losses_arr, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("DSM loss (eps-param)")
ax.set_title(f"Phase 3.10b — BBA GraphTransformer  (run {RUN_ID})")
ax.grid(True, alpha=0.3)
plt.tight_layout()

LOSS_PNG = "/content/310_loss_curve.png"
plt.savefig(LOSS_PNG, dpi=120)
# Immediately copy to Drive so if save-results cell never runs we still have it.
shutil.copy2(LOSS_PNG, os.path.join(DRIVE_RUN_DIR, "310_loss_curve.png"))
plt.show()
print(f"Saved to {LOSS_PNG} and Drive run dir.")

## 9. Score recovery evaluation

Cosine similarity between model predictions and precomputed `s_true` targets
on the last 10% of repeat r00 (held out — not shuffled into training epochs).

With `eps_parameterization`, the model predicts `v_h_unit`; we compare
direction-only (same as training loss).

In [ ]:
data_r00 = np.load(local_files[0])
x_t_all    = data_r00["x_t"].astype(np.float32)    # (N, n, d)
s_true_all = data_r00["s_true"].astype(np.float32)  # (N, n, d)
t_all      = data_r00["t"].astype(np.float32)        # (N,)

N_held = max(512, x_t_all.shape[0] // 10)
x_heldout = jnp.array(x_t_all[-N_held:].reshape(N_held, -1))    # (N_held, nd)
s_heldout = jnp.array(s_true_all[-N_held:].reshape(N_held, -1)) # (N_held, nd)
t_heldout = jnp.array(t_all[-N_held:, None])                    # (N_held, 1)

print(f"Held-out frames : {N_held}")

# Batch inference with EMA params
score_fn = lambda x, t: model.apply(state["ema_params"], x, t)
EVAL_B = 256
preds = []
for i in range(0, N_held, EVAL_B):
    preds.append(score_fn(x_heldout[i:i+EVAL_B], t_heldout[i:i+EVAL_B]))
s_pred = jnp.concatenate(preds, axis=0)  # (N_held, nd)

def cos_sim_batch(a, b):
    a_n = a / (jnp.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b_n = b / (jnp.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return float((a_n * b_n).sum(axis=-1).mean())

cos_overall = cos_sim_batch(s_pred, s_heldout)
print(f"\ncos_sim (all t) = {cos_overall:.4f}")
print(f"  PASS (>0.95)" if cos_overall > 0.95 else f"  below Phase 3.10b gate (0.95) — check loss curve and per-t breakdown")

# Per-t-bin breakdown — denser binning matches the success-gate spec
# (CURRENT_STATE.md: t ∈ {0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99}).
t_vals = np.array(t_heldout).flatten()
per_t_results = {}
print("\nPer-t-bin cos_sim:")
for lo, hi in [(0.0, 0.05), (0.05, 0.1), (0.1, 0.2), (0.2, 0.4),
               (0.4, 0.6), (0.6, 0.8), (0.8, 0.95), (0.95, 1.0)]:
    mask = (t_vals >= lo) & (t_vals < hi)
    n = int(mask.sum())
    if n == 0:
        continue
    c = cos_sim_batch(s_pred[mask], s_heldout[mask])
    per_t_results[f"{lo:.2f}-{hi:.2f}"] = {"n": n, "cos_sim": c}
    print(f"  t [{lo:.2f},{hi:.2f}]  n={n:5d}  cos_sim={c:.4f}")

## 10. Save results to Drive

In [ ]:
# ── Save *everything* needed for offline analysis to Drive ───────────────────
# Layout written:
#   DRIVE_RUN_DIR/
#     ├── env_info.json                      (written before training)
#     ├── config.json                        (written before training)
#     ├── loss_history.json                  (written at end of training cell)
#     ├── eval.json                          (here)
#     ├── 310_bba_gt_results.npz             (here — bundled arrays for plotting)
#     ├── 310_loss_curve.png                 (here)
#     └── checkpoints/
#         ├── ckpt_epoch_NNNNNN.npz          (synced incrementally during training)
#         └── ckpt_final.npz                 (saved by train_from_precomputed)
#
# Anything written to LOCAL_CKPT_DIR but missed by the incremental sync is
# copied here at the end.

import shutil, json, glob

# ── 1. Eval metrics → JSON (human-readable) and NPZ (numpy-loadable) ──────────
eval_summary = {
    "cos_sim_overall": float(cos_overall),
    "n_held"         : int(N_held),
    "per_t_bin"      : per_t_results,
    "final_loss"     : float(losses_v[-1]),
    "first_loss"     : float(losses_v[0]),
    "n_epochs_trained": int(epochs_v[-1]),
    "training_wall_min": float(t_wall / 60.0),
    "n_params"       : int(n_params),
    "gate_phase_3_10b": "PASS" if cos_overall > 0.95 else "FAIL",
}
eval_json_path = os.path.join(DRIVE_RUN_DIR, "eval.json")
with open(eval_json_path, "w") as f:
    json.dump(eval_summary, f, indent=2)
print(f"eval.json → Drive ({eval_summary['gate_phase_3_10b']})")

# Bundled numpy archive for fast offline plotting.
RESULTS_NPZ = os.path.join(DRIVE_RUN_DIR, "310_bba_gt_results.npz")
np.savez(
    RESULTS_NPZ,
    epochs            = np.array([e for e, _ in loss_history]),
    losses            = np.array([l for _, l in loss_history]),
    cos_sim_overall   = np.array([cos_overall]),
    n_params          = np.array([n_params]),
    t_held            = np.array(t_heldout).flatten(),
    config_str        = np.array([json.dumps(config_info)]),
    env_str           = np.array([json.dumps(env_info)]),
)
print(f"310_bba_gt_results.npz → Drive")

# ── 2. Loss curve PNG ─────────────────────────────────────────────────────────
LOSS_PNG_DRIVE = os.path.join(DRIVE_RUN_DIR, "310_loss_curve.png")
shutil.copy2(LOSS_PNG, LOSS_PNG_DRIVE)
print(f"310_loss_curve.png → Drive")

# ── 3. Checkpoint catch-up — copy anything the background sync missed ─────────
drive_ckpt_dir = os.path.join(DRIVE_RUN_DIR, "checkpoints")
os.makedirs(drive_ckpt_dir, exist_ok=True)
local_ckpts = sorted(glob.glob(os.path.join(LOCAL_CKPT_DIR, "ckpt_*.npz")))
n_copied = 0
for src in local_ckpts:
    dst = os.path.join(drive_ckpt_dir, os.path.basename(src))
    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
        shutil.copy2(src, dst)
        n_copied += 1
print(f"Checkpoints: {len(local_ckpts)} on local SSD, {n_copied} newly copied to Drive.")

drive_ckpts = sorted(os.listdir(drive_ckpt_dir))
print(f"  Drive checkpoints ({len(drive_ckpts)}):")
for c in drive_ckpts[:3]:
    print(f"    {c}")
if len(drive_ckpts) > 6:
    print(f"    … ({len(drive_ckpts) - 6} more)")
for c in drive_ckpts[-3:]:
    print(f"    {c}")

# ── 4. Final summary ──────────────────────────────────────────────────────────
print(f"\n✅ All artefacts persisted to:\n  {DRIVE_RUN_DIR}")
print(f"\nManifest:")
for entry in sorted(os.listdir(DRIVE_RUN_DIR)):
    full = os.path.join(DRIVE_RUN_DIR, entry)
    if os.path.isfile(full):
        sz_mb = os.path.getsize(full) / (1024 * 1024)
        print(f"  {entry:<35s}  {sz_mb:7.2f} MB")
    else:
        n = len(os.listdir(full))
        print(f"  {entry}/  ({n} files)")

## 11. Resume from checkpoint (if runtime was interrupted)

Run this cell **instead of** cell 7 if a checkpoint already exists on Drive.

**How to pick the resume source**: every prior training run is at
`DRIVE_OUT_DIR/run_<UTC_TS>/checkpoints/`. Set `RESUME_FROM_RUN` below to the
run-id you want to continue, or leave as `None` to autodetect the most recent
run-dir with at least one checkpoint.

In [ ]:
import re, glob
from training.train_manifold import load_checkpoint

# Set to a specific run-id (e.g. "20260629T120000Z") to resume that run, or
# leave None to pick up from the most recent run with checkpoints on Drive.
RESUME_FROM_RUN = None

# ── Locate the source checkpoint dir on Drive ─────────────────────────────────
if RESUME_FROM_RUN is None:
    candidates = sorted([
        d for d in glob.glob(os.path.join(DRIVE_OUT_DIR, "run_*"))
        if os.path.isdir(os.path.join(d, "checkpoints"))
        and len(os.listdir(os.path.join(d, "checkpoints"))) > 0
    ])
    if not candidates:
        # Backward compat: legacy layout (DRIVE_OUT_DIR/checkpoints/)
        if os.path.isdir(os.path.join(DRIVE_OUT_DIR, "checkpoints")):
            candidates = [DRIVE_OUT_DIR]
    if not candidates:
        raise FileNotFoundError("No prior runs with checkpoints found on Drive.")
    src_dir = candidates[-1]
    print(f"Auto-resuming from: {src_dir}")
else:
    src_dir = os.path.join(DRIVE_OUT_DIR, f"run_{RESUME_FROM_RUN}")
    if not os.path.isdir(src_dir):
        raise FileNotFoundError(f"No run dir at {src_dir}")
    print(f"Resuming from explicit run: {src_dir}")

drive_ckpt_dir_src = os.path.join(src_dir, "checkpoints")
if os.path.isdir(LOCAL_CKPT_DIR):
    shutil.rmtree(LOCAL_CKPT_DIR)
shutil.copytree(drive_ckpt_dir_src, LOCAL_CKPT_DIR)
print(f"Restored {len(os.listdir(LOCAL_CKPT_DIR))} checkpoints to {LOCAL_CKPT_DIR}")

# ── Pick latest checkpoint ────────────────────────────────────────────────────
ckpts = sorted(glob.glob(os.path.join(LOCAL_CKPT_DIR, "ckpt_epoch_*.npz")))
final_ckpt = os.path.join(LOCAL_CKPT_DIR, "ckpt_final.npz")

if os.path.exists(final_ckpt):
    print("Final checkpoint found — training was complete. Load params below if needed for eval.")
elif not ckpts:
    print("No checkpoints found — run cell 7 to start training from scratch.")
else:
    latest = ckpts[-1]
    m = re.search(r'epoch_(\d+)', os.path.basename(latest))
    done = int(m.group(1)) if m else 0
    remaining = N_EPOCHS - done
    print(f"Latest : {os.path.basename(latest)}  ({done} epochs done, {remaining} remaining)")

    example_x = np.zeros((1, N_ATOMS, D), dtype=np.float32)
    params_loaded = load_checkpoint(latest, model, manifold, sde, example_x)
    print("Params loaded OK")

    # Continue cosine schedule from where it left off (scaled by remaining/total).
    lr_resume = LR * remaining / N_EPOCHS
    print(f"Resuming: {remaining} epochs, lr={lr_resume:.2e}")
    print("NOTE: to persist results to a NEW timestamped run dir on Drive, re-run cell 7 instead.")

    state, loss_history = train_from_precomputed(
        model=model, manifold=manifold, sde=sde,
        precomputed_dir=LOCAL_DATA_DIR,
        n_epochs=remaining, batch_size=BATCH_SIZE,
        learning_rate=lr_resume, grad_clip=GRAD_CLIP,
        ema_decay=EMA_DECAY, eps_parameterization=EPS_PARAM,
        seed=SEED, log_every=LOG_EVERY, ckpt_dir=LOCAL_CKPT_DIR,
    )

---
## Notes

### Expected results
- **Loss**: ~22 at epoch 50, decreasing to <5 by epoch 3000 with `eps_parameterization=True`
- **cos_sim held-out**: >0.9 after 3000 epochs (D16 showed 0.977 at N=128/300ep)
- **Runtime**: ~11 min on A100/H100

### If cos_sim is low (<0.7) after 3000 epochs
1. Check loss curve — still decreasing? Train longer (increase `N_EPOCHS`).
2. Check per-t-bin: if only high-t bins are low, the model learned well; high-t score is near-zero by design.
3. Verify precomputed data: regenerate with `scripts/precompute_noised_data.py --protein bba --n-repeats 10`.

### Key design choices
- **`eps_parameterization=True`**: predicts `v_h_unit` (Euclidean norm ≈ 2.81, flat across all t).
  Eliminates the 200× amplitude variation in `s_true` that caused the zero-output basin.
  See `TRAINING_COLLAPSE_ANALYSIS.md`.
- **`ALPHA=1.0`**: Both BBA and AK use δ=1.0. The ShapeManifold constructor default (0.1) is never used for actual proteins.
- **No `base_point`** in `ShapeManifold`: not needed for training (only for `s_exp`/`s_geodesic`
  alignment during inference). Precomputed targets were generated with a base point.

### Next steps after Phase 3.10
1. Reverse SDE sampling: `ManifoldEulerMaruyama` in reverse mode, 500 steps, 500 samples
2. JS divergence on pairwise Cα distances (generated vs training set)
3. Phase 4: derive the Riemannian FP residual from Kolmogorov forward equation on (M, g)